# 05 — Forecasting

**Phenomenological Growth Models Lab**

Everything so far has been about describing data you already have. Forecasting is
harder, and the honesty bar is higher: the only way to know whether a forecast was good
is to compare it against data the model never saw.

Assumes [03 — Uncertainty](03_uncertainty.ipynb).

## Setup

These notebooks run against the environment pinned in this repository's `Project.toml`
and `Manifest.toml`. The cell below activates it and installs anything missing.

The first run downloads and precompiles the dependencies — a few minutes, once.
Every run after that starts in seconds.

Because the versions are pinned in `Manifest.toml`, you get the same package versions
these notebooks were written against, whatever else is installed on your machine. If you
skip this cell and rely on your global environment, newer versions may resolve
differently and the numbers below may not match.

In [ ]:
using Pkg

# Works whether the notebook was launched from the repository root or from notebooks/.
root = isfile("Project.toml") ? "." : ".."
Pkg.activate(root)
Pkg.instantiate()

println("Active project: ", Base.active_project())

In [ ]:
using GrowthFit
using Plots, Statistics, Printf, DelimitedFiles, Random

default(size = (760, 420), lw = 2, legend = :topright, framestyle = :box,
        guidefontsize = 10, titlefontsize = 11,
        left_margin = 4Plots.mm, bottom_margin = 4Plots.mm)

# Threads. Notebooks 03 and 05 bootstrap in parallel. A Jupyter kernel does
# NOT inherit `-t` from whatever launched Jupyter — the thread count is fixed
# by the kernel itself — so if this says 1, see the README for the one-off
# `IJulia.installkernel` step. Results are identical either way; only speed
# changes.
println("GrowthFit loaded. Threads available: ", Threads.nthreads())
Threads.nthreads() == 1 && println("  (single-threaded — notebooks 03 and 05 will be slower; see README)");


> **Tip.** The bootstrap in notebooks 03 and 05 runs in parallel. If you want it faster,
> restart Julia with `-t 8`. Results are identical either way — each replicate is seeded
> independently of the thread count.

In [ ]:
function load_series(name)
    path = isfile(joinpath("data", name)) ? joinpath("data", name) :
           joinpath("..", "data", name)
    isfile(path) || error("Could not find $name. Run this notebook from inside the " *
                          "cloned repository so the data/ folder is reachable.")
    raw = readdlm(path, '\t', Float64)
    return raw[:, 1], raw[:, 2]          # time, incidence
end

timevect_all, ydata_all = load_series("jalisco_measles.txt")

@printf("%d weeks, peak of %.0f cases, %.0f cases total\n",
        length(ydata_all), maximum(ydata_all), sum(ydata_all))

In [ ]:
calibration_period = 40
forecast_horizon   = 3
timevect = timevect_all[1:calibration_period]
ydata    = ydata_all[1:calibration_period]
I0 = ydata[1]

println("total weeks available: ", length(ydata_all))
println("calibrating on:        ", calibration_period)
println("evaluating on:         ", length(ydata_all) - calibration_period)

## One forecast

Fit on the calibration window, project forward, then reveal the held-out weeks.

In [ ]:
Random.seed!(1)
fit  = fit_growth_model(:grm, timevect, ydata; dist = :nb1, n_restarts = 10)
boot = run_bootstrap(:grm, timevect, ydata, fit;
                     dist = :nb1, M = 300,
                     forecast_horizon = forecast_horizon, seed = 20260726)

fig = plot_fit_and_forecast(boot, timevect, ydata, ydata_all, timevect_all;
                            forecast_horizon = forecast_horizon)

`plot_fit_and_forecast` is the package's built-in panel. Building it by hand is worth
doing once, because it makes explicit what is being plotted:

In [ ]:
nT = size(boot.forecast_noisy, 1)
tf = timevect_all[1:nT]

m  = [median(boot.forecast_noisy[i, :]) for i in 1:nT]
lo = [quantile(boot.forecast_noisy[i, :], 0.025) for i in 1:nT]
hi = [quantile(boot.forecast_noisy[i, :], 0.975) for i in 1:nT]

plot(tf, m, ribbon = (m .- lo, hi .- m), fillalpha = 0.25,
     label = "median + 95% PI", lc = :crimson, fc = :crimson,
     xlabel = "week", ylabel = "weekly cases", title = "3-week-ahead forecast")
scatter!(timevect, ydata, label = "calibration", mc = :steelblue, ms = 4, msw = 0)
scatter!(timevect_all[calibration_period+1:nT], ydata_all[calibration_period+1:nT],
         label = "held out (unseen)", mc = :orange, ms = 6, msw = 0, marker = :diamond)
vline!([timevect[end] + 0.5], label = "forecast origin", ls = :dash, lc = :grey40)

## Scoring the forecast

Same metrics as notebook 03, now applied to data the model never saw. This is the only
number that means anything about forecast skill.

In [ ]:
function interval_score(y, l, u, α)
    s = (u - l)
    y < l && (s += 2 / α * (l - y))
    y > u && (s += 2 / α * (y - u))
    return s
end

# Eleven α levels — the forecast-hub convention, and what the package uses.
function wis(y, sample; αs = vcat(0.02, 0.05, collect(0.1:0.1:0.9)))
    m, K, total = median(sample), length(αs), 0.0
    total = 0.5 * abs(y - m)
    for α in αs
        total += (α / 2) * interval_score(y, quantile(sample, α/2),
                                          quantile(sample, 1 - α/2), α)
    end
    return total / (K + 0.5)
end

# Point forecast from forecast_curves (model trajectories, no observation
# noise); intervals and WIS from forecast_noisy. Scoring the point forecast
# against the noisy curves would measure the model against a random
# realization of itself. This matches what the package does.
println("step   observed   predicted      95% PI        covered     WIS")
println("-"^66)
for h in 1:forecast_horizon
    i = calibration_period + h
    i > length(ydata_all) && break
    curve = boot.forecast_curves[i, :]
    noisy = boot.forecast_noisy[i, :]
    y = ydata_all[i]
    l, u = quantile(noisy, 0.025), quantile(noisy, 0.975)
    @printf("%3d %10.0f %11.1f  [%5.0f, %5.0f] %10s %7.2f\n",
            h, y, median(curve), l, u, (l <= y <= u ? "yes" : "NO"), wis(y, noisy))
end

The package computes the same summary:

In [ ]:
fc = compute_forecast_performance(boot, ydata_all, calibration_period, forecast_horizon)

# A Vector of NamedTuples, one per horizon step — so ask the first element
# for its fields, not the vector.
println(keys(fc[1]))
fc

## Skill decays with horizon

A one-week-ahead forecast is a much easier problem than a four-week-ahead one. Quantify
that rather than assuming it.

In [ ]:
Random.seed!(1)
H = min(5, length(ydata_all) - calibration_period)
boot_H = run_bootstrap(:grm, timevect, ydata, fit;
                       dist = :nb1, M = 300, forecast_horizon = H, seed = 20260726)

hs, maes, wiss, widths, relwidths = Int[], Float64[], Float64[], Float64[], Float64[]
for h in 1:H
    i = calibration_period + h
    i > length(ydata_all) && break
    curve = boot_H.forecast_curves[i, :]   # point forecast
    s     = boot_H.forecast_noisy[i, :]    # predictive distribution
    y = ydata_all[i]
    push!(hs, h)
    push!(maes, abs(y - median(curve)))
    push!(wiss, wis(y, s))
    w = quantile(s, 0.975) - quantile(s, 0.025)
    push!(widths, w)
    push!(relwidths, w / max(median(curve), 1e-8))   # width relative to the prediction
end

p1 = plot(hs, maes, marker = :circle, label = "absolute error",
          xlabel = "horizon (weeks)", ylabel = "cases", title = "Point error")
p2 = plot(hs, wiss, marker = :circle, label = "WIS", lc = :crimson,
          xlabel = "horizon (weeks)", ylabel = "WIS", title = "Weighted interval score")
p3 = plot(hs, widths, marker = :circle, label = "95% PI width", lc = :seagreen,
          xlabel = "horizon (weeks)", ylabel = "cases", title = "Interval width (cases)")
p4 = plot(hs, relwidths, marker = :circle, label = "width / prediction", lc = :purple,
          xlabel = "horizon (weeks)", ylabel = "ratio", title = "Interval width (relative)")
plot(p1, p2, p3, p4, layout = (2, 2), size = (900, 620))

Read those four panels carefully, because the obvious reading of the third one is wrong.

**Point error** grows with horizon, as expected.

**WIS** does not move monotonically. With only three points that is noise rather than a
pattern — one forecast happening to land in a well-covered part of the predictive
distribution is enough to produce a dip. Do not read a trend into three numbers; the
rolling-origin evaluation below is what gives you enough forecasts to say anything.

**Interval width in cases shrinks** with horizon. It is tempting to call that
overconfidence — the model claiming *more* certainty further out — but it is not. We are
on the declining tail of the epidemic, so predicted counts are falling fast, and under NB1
the variance scales with the mean. An interval around a prediction of 15 cases is
necessarily narrower in absolute terms than one around 26, however uncertain the model is.

**Interval width relative to the prediction** is the honest measure, and it does grow.
That is the model being appropriately less certain further out.

The general lesson: on count data with mean-dependent variance, absolute interval width
conflates uncertainty with the level of the series. Scale it by the prediction before
interpreting it.

In [ ]:
Random.seed!(1)

origins = 25:2:39
h1_err, h1_wis, h1_cov, used = Float64[], Float64[], Bool[], Int[]

for orig in origins
    orig + 1 > length(ydata_all) && continue
    tv, yd = timevect_all[1:orig], ydata_all[1:orig]
    f = fit_growth_model(:grm, tv, yd; dist = :nb1, n_restarts = 5)
    f.converged || continue
    b = run_bootstrap(:grm, tv, yd, f; dist = :nb1, M = 100,
                      forecast_horizon = 1, seed = 20260726)
    curve = b.forecast_curves[orig + 1, :]
    s = b.forecast_noisy[orig + 1, :]
    y = ydata_all[orig + 1]
    l, u = quantile(s, 0.025), quantile(s, 0.975)
    push!(used, orig)
    push!(h1_err, abs(y - median(curve)))
    push!(h1_wis, wis(y, s))
    push!(h1_cov, l <= y <= u)
    @printf("origin week %2d: observed %4.0f, predicted %6.1f, WIS %6.2f %s\n",
            Int(timevect_all[orig]), y, median(curve), wis(y, s), (l <= y <= u ? "" : "  <- missed"))
end

println()
@printf("mean 1-week MAE : %.2f\n", mean(h1_err))
@printf("mean 1-week WIS : %.2f\n", mean(h1_wis))
@printf("coverage        : %.0f%% of %d forecasts\n", 100mean(h1_cov), length(h1_cov))

In [ ]:
plot(timevect_all[used], h1_wis, marker = :circle, label = "WIS",
     xlabel = "forecast origin (week)", ylabel = "WIS",
     title = "One-week-ahead skill over the epidemic")
scatter!(twinx(), timevect_all, ydata_all, label = "cases", mc = :grey70,
         ms = 3, msw = 0, ylabel = "weekly cases", legend = :topright)

Look at where the WIS spikes. Forecasts are typically worst around the **turning
point** — the model has not yet seen enough of the decline to know the epidemic is
peaking, so it keeps projecting growth. That is a structural limitation of fitting a
single-wave curve, not a failure of this particular implementation.

It also explains why forecast hubs combine many models: they fail at different times.

## Honest reporting

If you take one thing from this lab:

- **Never evaluate on data you fitted.** In-sample fit is not forecast skill.
- **Report intervals, not point estimates.** And say what they cover.
- **Score with a proper scoring rule.** MAE alone rewards overconfidence.
- **State the assumptions.** These intervals assume the model form is correct. It is not.
- **Say when the model does not apply.** Multi-wave epidemics, changing interventions,
  and reporting artifacts all break the single-curve assumption.

---

## Exercises

1. Run the rolling-origin loop for the Generalized Logistic and compare mean WIS with
   the Generalized Richards. Does the extra parameter earn its place out of sample?
   (Notebook 06 runs this comparison across the whole epidemic.)

2. Change the horizon to 2 and re-run. How much does mean WIS increase?

3. Take the mean of the GRM and GLM predictive samples at each origin and score that as
   an ensemble. Does it beat both components? (It often does — that is why ensembles
   dominate forecasting competitions.)

4. Repeat on the Mexico City series. Is one-week-ahead skill better or worse, and does
   the size of the outbreak explain the difference?

---

**Next:** [06 — Rolling windows](06_rolling_windows.ipynb), where every model is
fitted and scored week by week across the whole outbreak.